<a href="https://colab.research.google.com/github/KUNALDEBNATH/DL_REV2/blob/main/DL_Review2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import torchvision.models as models
import torchvision.datasets as datasets
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


# ===========================================================
# GLOBAL SETTINGS
# ===========================================================

SEED          = 42
NUM_CLASSES   = 10
SEQ_LEN       = 4
BATCH_SIZE    = 32
EPOCHS        = 5
EMBED_DIM     = 256
HIDDEN_DIM    = 256
NUM_LAYERS    = 2
DROPOUT       = 0.3
LEARNING_RATE = 0.001

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# ===========================================================
# CRITERION 1 — TEMPORAL DATA PREPROCESSING PIPELINE
# Marks: 2
# What is looked for: Construction of a pipeline for
# handling temporal or sequential data.
#
# We download MNIST and group images of the same digit
# into sequences of length SEQ_LEN.
# Each sequence = one training sample.
# Images are resized to 32x32 and converted to 3 channels.
# ===========================================================

class MNISTSequenceDataset(Dataset):

    def __init__(self, train=True):

        self.transform = transforms.Compose([
            transforms.Resize((32, 32)),
            transforms.Grayscale(num_output_channels=3),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

        self.mnist = datasets.MNIST(
            root="./data", train=train,
            download=True, transform=self.transform
        )

        digit_to_indices = {i: [] for i in range(10)}
        for index, (_, label) in enumerate(self.mnist):
            digit_to_indices[label].append(index)

        self.sequences = []
        for digit, indices in digit_to_indices.items():
            for start in range(0, len(indices) - SEQ_LEN, SEQ_LEN):
                frame_indices = indices[start : start + SEQ_LEN]
                self.sequences.append((frame_indices, digit))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        frame_indices, label = self.sequences[index]
        frames = torch.stack([self.mnist[i][0] for i in frame_indices])
        return frames, label


def get_dataloaders():
    print("  Downloading and preparing MNIST sequences...")

    train_val_dataset = MNISTSequenceDataset(train=True)
    test_dataset      = MNISTSequenceDataset(train=False)

    total      = len(train_val_dataset)
    val_size   = int(0.15 * total)
    train_size = total - val_size

    train_dataset, val_dataset = random_split(
        train_val_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

    print(f"  Train: {train_size} | Val: {val_size} | Test: {len(test_dataset)} sequences")
    return train_loader, val_loader, test_loader


def visualise_samples(train_loader):
    frames, labels = next(iter(train_loader))
    fig, axes = plt.subplots(3, SEQ_LEN, figsize=(SEQ_LEN * 2, 7))
    for row in range(3):
        for col in range(SEQ_LEN):
            img = frames[row, col].permute(1, 2, 0).numpy()
            img = (img - img.min()) / (img.max() - img.min())
            axes[row, col].imshow(img[:, :, 0], cmap="gray")
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_title(f"Digit: {labels[row].item()}", fontsize=9)
    plt.suptitle("Sample MNIST Sequences (each row = one sequence)", fontweight="bold")
    plt.tight_layout()
    plt.savefig("sample_sequences.png", dpi=150)
    plt.close()
    print("  [Saved] sample_sequences.png")


# ===========================================================
# CRITERION 2 — FEATURE EXTRACTION USING PRETRAINED CNN MODELS
# Marks: 3
# What is looked for: Correct use of at least two
# pretrained architectures.
#
# We use ResNet-18 and MobileNet-V2 (both pretrained on ImageNet).
# All weights are FROZEN — we only extract features, not train.
# Input:  (Batch, Frames, 3, H, W)
# Output: (Batch, Frames, feature_size)
# ===========================================================

class CNNFeatureExtractor(nn.Module):

    def __init__(self, backbone_name="resnet18"):
        super().__init__()

        if backbone_name == "resnet18":
            base_model       = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            self.feature_dim = 512
            self.encoder     = nn.Sequential(*list(base_model.children())[:-1])

        elif backbone_name == "mobilenet_v2":
            base_model       = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
            self.feature_dim = 1280
            self.encoder     = base_model.features

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        for param in self.encoder.parameters():
            param.requires_grad = False

    def forward(self, x):
        batch_size, num_frames, C, H, W = x.shape
        all_frames  = x.view(batch_size * num_frames, C, H, W)
        frame_feats = self.pool(self.encoder(all_frames))
        frame_feats = frame_feats.view(batch_size, num_frames, self.feature_dim)
        return frame_feats


# ===========================================================
# CRITERION 3 — FINE-TUNING PRETRAINED CNN MODELS
# Marks: 3
# What is looked for: Demonstration of transfer learning by
# unfreezing selected layers and retraining on the task dataset.
#
# We use ResNet-18 but unfreeze only layer4.
# This means layer4 will adapt to MNIST during training
# while all earlier layers stay frozen (transfer learning).
# ===========================================================

class FineTunedResNet(nn.Module):

    def __init__(self):
        super().__init__()
        self.feature_dim = 512

        base_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        for param in base_model.parameters():
            param.requires_grad = False

        for param in base_model.layer4.parameters():
            param.requires_grad = True

        self.encoder = nn.Sequential(*list(base_model.children())[:-1])
        self.pool    = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        batch_size, num_frames, C, H, W = x.shape
        all_frames  = x.view(batch_size * num_frames, C, H, W)
        frame_feats = self.pool(self.encoder(all_frames))
        frame_feats = frame_feats.view(batch_size, num_frames, self.feature_dim)
        return frame_feats


# ===========================================================
# CRITERION 4 — EMBEDDING USAGE
# Marks: 2
# What is looked for: Effective use of embeddings as
# inputs to sequence models.
#
# Projects CNN features into a fixed-size embedding vector.
# Also adds positional embeddings so the model knows
# the order of frames (frame 1, frame 2, frame 3...).
# ===========================================================

class TemporalEmbedding(nn.Module):

    def __init__(self, input_dim, embed_dim):
        super().__init__()
        self.linear    = nn.Linear(input_dim, embed_dim)
        self.pos_embed = nn.Embedding(64, embed_dim)
        self.norm      = nn.LayerNorm(embed_dim)

    def forward(self, x):
        batch_size, num_frames, _ = x.shape
        positions      = torch.arange(num_frames, device=x.device).unsqueeze(0)
        projected      = self.linear(x)
        with_positions = projected + self.pos_embed(positions)
        output         = self.norm(with_positions)
        return output


# ===========================================================
# CRITERION 5 — ATTENTION-BASED MODEL
# Marks: 2
# What is looked for: Implementation or integration of an
# attention mechanism to improve sequence modeling.
#
# Scaled dot-product attention over the time (frame) axis.
# Lets the model assign higher importance to frames that
# are most useful for identifying the digit.
# ===========================================================

class TemporalAttention(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key   = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.scale = hidden_dim ** 0.5

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        attention_scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        attention_weights = torch.softmax(attention_scores, dim=-1)
        attended          = torch.bmm(attention_weights, V)
        output            = attended.mean(dim=1)
        return output


# ===========================================================
# CRITERION 6 — RNN / LSTM / GRU IMPLEMENTATION
# Marks: 4
# What is looked for: Correct implementation of sequence
# models (RNN, LSTM, or GRU) to capture temporal dependencies.
#
# Full pipeline:
#   Frames → CNN → Embedding → RNN/LSTM/GRU → Attention → Label
#
# The rnn_type parameter lets us switch between RNN, LSTM, GRU.
# ===========================================================

class SequenceClassifier(nn.Module):

    def __init__(self, backbone="resnet18", rnn_type="LSTM"):
        super().__init__()

        if backbone == "finetuned_resnet":
            self.cnn = FineTunedResNet()
        else:
            self.cnn = CNNFeatureExtractor(backbone)

        self.embedding = TemporalEmbedding(self.cnn.feature_dim, EMBED_DIM)

        rnn_options = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}
        self.rnn = rnn_options[rnn_type](
            input_size  = EMBED_DIM,
            hidden_size = HIDDEN_DIM,
            num_layers  = NUM_LAYERS,
            batch_first = True,
            dropout     = DROPOUT if NUM_LAYERS > 1 else 0.0
        )

        self.attention  = TemporalAttention(HIDDEN_DIM)

        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, NUM_CLASSES)
        )

    def forward(self, x):
        cnn_features  = self.cnn(x)
        embedded      = self.embedding(cnn_features)
        rnn_output, _ = self.rnn(embedded)
        context       = self.attention(rnn_output)
        logits        = self.classifier(context)
        return logits


# ===========================================================
# TRAINING & EVALUATION HELPERS
# (used by Criterion 7 and Criterion 8)
# ===========================================================

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for frames, labels in loader:
        frames = frames.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        predictions = model(frames)
        loss        = criterion(predictions, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct    += (predictions.argmax(1) == labels).sum().item()
        total      += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for frames, labels in loader:
        frames = frames.to(DEVICE)
        labels = labels.to(DEVICE)

        predictions = model(frames)
        loss        = criterion(predictions, labels)

        total_loss += loss.item() * labels.size(0)
        all_preds.extend(predictions.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(all_labels)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_preds, all_labels


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LEARNING_RATE):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history   = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        train_loss, train_acc   = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"    Epoch {epoch:02d}/{epochs} | "
              f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}")

    return history


# ===========================================================
# CRITERION 7 — HYPERPARAMETER EXPERIMENTATION
# Marks: 1
# What is looked for: Systematic exploration of at least two
# hyperparameters (e.g., learning rate, sequence length,
# batch size).
#
# We try 4 combinations:
#   learning_rate ∈ [0.001, 0.0005]
#   hidden_dim    ∈ [128, 256]
# Best combination is chosen by highest validation accuracy.
# ===========================================================

def hyperparameter_search(train_loader, val_loader):
    learning_rates = [0.001, 0.0005]
    hidden_sizes   = [128, 256]
    results        = []

    for lr in learning_rates:
        for hd in hidden_sizes:
            print(f"  Trying lr={lr}, hidden_dim={hd}")

            global HIDDEN_DIM
            HIDDEN_DIM = hd

            model    = SequenceClassifier().to(DEVICE)
            history  = train_model(model, train_loader, val_loader, epochs=2, lr=lr)
            best_acc = max(history["val_acc"])

            results.append({
                "lr":           lr,
                "hidden_dim":   hd,
                "best_val_acc": best_acc
            })
            print(f"  -> Best Val Acc: {best_acc:.4f}\n")

    results.sort(key=lambda r: r["best_val_acc"], reverse=True)
    HIDDEN_DIM = 256
    return results


# ===========================================================
# CRITERION 8 — MODEL COMPARISON & EVALUATION METRICS
# Marks: 2
# What is looked for: Comparison of RNN, LSTM, and GRU Models.
#
# We train all three models and compare using:
#   - Accuracy
#   - Precision, Recall, F1 Score (classification report)
#   - Confusion Matrix
# ===========================================================

def compare_models(train_loader, val_loader, test_loader):
    results = {}

    for rnn_type in ["RNN", "LSTM", "GRU"]:
        print(f"\n  Training {rnn_type} model...")

        model   = SequenceClassifier(rnn_type=rnn_type).to(DEVICE)
        history = train_model(model, train_loader, val_loader)

        _, test_acc, preds, labels = evaluate(model, test_loader, nn.CrossEntropyLoss())

        results[rnn_type] = {
            "history":  history,
            "test_acc": test_acc,
            "report":   classification_report(labels, preds, zero_division=0),
            "cm":       confusion_matrix(labels, preds),
        }

        print(f"\n  {rnn_type} Test Accuracy : {test_acc:.4f}")
        print(f"  {rnn_type} Classification Report:\n")
        print(results[rnn_type]["report"])

    return results


# ===========================================================
# CRITERION 9 — CODE ORGANISATION
# Marks: 1
# What is looked for: Well-structured repository or notebook
# with clear documentation, modular code, reproducible
# experiments, and proper comments.
#
# All plots are saved as .png files for reporting.
# ===========================================================

def plot_training_curves(results):
    colors = {"RNN": "steelblue", "LSTM": "darkorange", "GRU": "forestgreen"}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for name, data in results.items():
        epoch_range = range(1, len(data["history"]["val_acc"]) + 1)
        axes[0].plot(epoch_range, data["history"]["val_acc"],  label=name, color=colors[name])
        axes[1].plot(epoch_range, data["history"]["val_loss"], label=name, color=colors[name])

    axes[0].set(title="Validation Accuracy", xlabel="Epoch", ylabel="Accuracy")
    axes[1].set(title="Validation Loss",     xlabel="Epoch", ylabel="Loss")
    for ax in axes:
        ax.legend(); ax.grid(True)

    plt.suptitle("MNIST — RNN vs LSTM vs GRU Training Curves", fontweight="bold")
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.close()
    print("  [Saved] training_curves.png")


def plot_confusion_matrices(results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (name, data) in zip(axes, results.items()):
        sns.heatmap(
            data["cm"], annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=range(NUM_CLASSES), yticklabels=range(NUM_CLASSES)
        )
        ax.set(
            title=f"{name}  (Acc = {data['test_acc']:.3f})",
            xlabel="Predicted", ylabel="True"
        )
    plt.tight_layout()
    plt.savefig("confusion_matrices.png", dpi=150)
    plt.close()
    print("  [Saved] confusion_matrices.png")


def plot_accuracy_bar(results):
    colors = {"RNN": "steelblue", "LSTM": "darkorange", "GRU": "forestgreen"}
    names  = list(results.keys())
    accs   = [results[n]["test_acc"] for n in names]

    plt.figure(figsize=(6, 4))
    bars = plt.bar(names, accs, color=[colors[n] for n in names], edgecolor="black")
    plt.bar_label(bars, fmt="%.3f", padding=3)
    plt.ylim(0, 1.15)
    plt.title("Test Accuracy Comparison: RNN vs LSTM vs GRU")
    plt.ylabel("Accuracy")
    plt.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.savefig("model_accuracy_bar.png", dpi=150)
    plt.close()
    print("  [Saved] model_accuracy_bar.png")


# ===========================================================
# MAIN — Runs all criteria in order
# ===========================================================

def main():

    print("\n" + "="*55)
    print("  CRITERION 1 — Temporal Data Preprocessing Pipeline")
    print("="*55)
    train_loader, val_loader, test_loader = get_dataloaders()
    visualise_samples(train_loader)

    print("\n" + "="*55)
    print("  CRITERION 2 — Feature Extraction (ResNet-18 + MobileNet-V2)")
    print("="*55)
    for backbone in ["resnet18", "mobilenet_v2"]:
        extractor   = CNNFeatureExtractor(backbone).to(DEVICE)
        dummy_input = torch.randn(2, SEQ_LEN, 3, 32, 32).to(DEVICE)
        with torch.no_grad():
            features = extractor(dummy_input)
        print(f"  {backbone:15s} -> output shape: {tuple(features.shape)}")

    print("\n" + "="*55)
    print("  CRITERION 3 — Fine-Tuning ResNet-18 (layer4 unfrozen)")
    print("="*55)
    ft_model         = FineTunedResNet().to(DEVICE)
    trainable_params = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
    total_params     = sum(p.numel() for p in ft_model.parameters())
    print(f"  Trainable params: {trainable_params:,} / {total_params:,} "
          f"({100 * trainable_params / total_params:.1f}%)")

    print("\n" + "="*55)
    print("  CRITERION 4 — Embedding Usage")
    print("="*55)
    sample_cnn_output = torch.randn(2, SEQ_LEN, 512).to(DEVICE)
    embedding_layer   = TemporalEmbedding(input_dim=512, embed_dim=EMBED_DIM).to(DEVICE)
    embedded_output   = embedding_layer(sample_cnn_output)
    print(f"  CNN output shape    : {tuple(sample_cnn_output.shape)}")
    print(f"  Embedded output shape: {tuple(embedded_output.shape)}")

    print("\n" + "="*55)
    print("  CRITERION 5 — Attention-Based Model")
    print("="*55)
    sample_rnn_output = torch.randn(2, SEQ_LEN, HIDDEN_DIM).to(DEVICE)
    attention_layer   = TemporalAttention(hidden_dim=HIDDEN_DIM).to(DEVICE)
    attention_output  = attention_layer(sample_rnn_output)
    print(f"  RNN output shape      : {tuple(sample_rnn_output.shape)}")
    print(f"  After attention shape : {tuple(attention_output.shape)}")

    print("\n" + "="*55)
    print("  CRITERION 6 — RNN / LSTM / GRU Implementation")
    print("="*55)
    print("  (Handled inside SequenceClassifier — trained in Criterion 8)")

    print("\n" + "="*55)
    print("  CRITERION 7 — Hyperparameter Experimentation")
    print("="*55)
    hp_results = hyperparameter_search(train_loader, val_loader)
    best = hp_results[0]
    print(f"  Best config — lr: {best['lr']}  hidden_dim: {best['hidden_dim']}  "
          f"val_acc: {best['best_val_acc']:.4f}")

    print("\n" + "="*55)
    print("  CRITERION 8 — Model Comparison & Evaluation Metrics")
    print("="*55)
    comparison = compare_models(train_loader, val_loader, test_loader)

    print("\n" + "="*55)
    print("  CRITERION 9 — Code Organisation (Saving All Plots)")
    print("="*55)
    plot_training_curves(comparison)
    plot_confusion_matrices(comparison)
    plot_accuracy_bar(comparison)

    print("\n" + "="*55)
    print("  FINAL RESULTS SUMMARY")
    print("="*55)
    print(f"  {'Model':<10}  {'Test Accuracy':>13}")
    print("  " + "-"*26)
    for name, data in comparison.items():
        print(f"  {name:<10}  {data['test_acc']:>13.4f}")
    best_model = max(comparison, key=lambda k: comparison[k]["test_acc"])
    print(f"\n  Best Model: {best_model}  ({comparison[best_model]['test_acc']:.4f})")
    print("="*55)


if __name__ == "__main__":
    main()

Using device: cuda

  CRITERION 1 — Temporal Data Preprocessing Pipeline


100%|██████████| 9.91M/9.91M [00:01<00:00, 4.98MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 128kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.07MB/s]


  Train: 12746 | Val: 2249 | Test: 2493 sequences
  [Saved] sample_sequences.png

  CRITERION 2 — Feature Extraction (ResNet-18 + MobileNet-V2)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 204MB/s]


  resnet18        -> output shape: (2, 4, 512)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 208MB/s]


  mobilenet_v2    -> output shape: (2, 4, 1280)

  CRITERION 3 — Fine-Tuning ResNet-18 (layer4 unfrozen)
  Trainable params: 8,393,728 / 11,176,512 (75.1%)

  CRITERION 4 — Embedding Usage
  CNN output shape    : (2, 4, 512)
  Embedded output shape: (2, 4, 256)

  CRITERION 5 — Attention-Based Model
  RNN output shape      : (2, 4, 256)
  After attention shape : (2, 256)

  CRITERION 6 — RNN / LSTM / GRU Implementation
  (Handled inside SequenceClassifier — trained in Criterion 8)

  CRITERION 7 — Hyperparameter Experimentation
  Trying lr=0.001, hidden_dim=128
    Epoch 01/2 | Train Loss: 0.6925  Acc: 0.7697 | Val Loss: 0.1875  Acc: 0.9364
    Epoch 02/2 | Train Loss: 0.2277  Acc: 0.9255 | Val Loss: 0.1153  Acc: 0.9627
  -> Best Val Acc: 0.9627

  Trying lr=0.001, hidden_dim=256
    Epoch 01/2 | Train Loss: 0.6483  Acc: 0.7755 | Val Loss: 0.2046  Acc: 0.9315
    Epoch 02/2 | Train Loss: 0.2231  Acc: 0.9292 | Val Loss: 0.1243  Acc: 0.9582
  -> Best Val Acc: 0.9582

  Trying lr=0.0005, 